In [10]:
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Quote class for structured data
class Quote:
    def __init__(self, symbol, price):
        self.symbol = symbol
        self.price = price

    def get(self, attr):
        return getattr(self, attr, None)

# Simulated Trading API class
class TradingAPI:
    def get_current_price(self, symbol: str) -> float:
        # Dummy fixed price
        return 100.0

    def get_quote(self, symbol: str) -> Quote:
        price = self.get_current_price(symbol)
        return Quote(symbol, price)

    def get_all_quotes(self, symbols: list) -> list:
        return [self.get_quote(symbol) for symbol in symbols]

    def buy_stock(self, symbol, quantity):
        price = self.get_current_price(symbol)
        print(f"[SIM] Buying {quantity} shares of {symbol} at {price}")
        return {"symbol": symbol, "quantity": quantity, "price": price, "side": "buy"}

    def sell_stock(self, symbol, quantity):
        price = self.get_current_price(symbol)
        print(f"[SIM] Selling {quantity} shares of {symbol} at {price}")
        return {"symbol": symbol, "quantity": quantity, "price": price, "side": "sell"}

# Instance of TradingAPI
api = TradingAPI()

def buy_stock(symbol, quantity):
    return api.buy_stock(symbol, quantity)

def sell_stock(symbol, quantity):
    return api.sell_stock(symbol, quantity)

def get_price(symbol):
    return api.get_current_price(symbol)

# Load Hugging Face LLM model
model_name = "Salesforce/codegen-350M-mono"  # ✅ Lightweight public model

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()
if torch.cuda.is_available():
    model.to("cuda")

# Function to generate code from user prompt
def generate_code(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=150)
    code = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return code

# Execute code safely
def safe_execute(code_str, local_env):
    try:
        exec_locals = {}
        exec(code_str, local_env, exec_locals)
        return exec_locals.get("result", "Executed Successfully")
    except Exception as e:
        return f"Error: {e}"

# Pipeline function for Gradio
def run_pipeline(user_prompt):
    code = generate_code(user_prompt)
    local_env = {
        "buy_stock": buy_stock,
        "sell_stock": sell_stock,
        "get_price": get_price
    }
    result = safe_execute(code, local_env)
    return code, result

# Gradio Interface
gr.Interface(
    fn=run_pipeline,
    inputs=gr.Textbox(label="Describe Trading Logic", lines=4),
    outputs=[
        gr.Code(label="Generated Trading Code"),
        gr.Textbox(label="Execution Output")
    ],
    title="AI Trading Code Generator",
    description="Generates Python trading logic using an open-source LLM."
).launch()

Some weights of the model checkpoint at Salesforce/codegen-350M-mono were not used when initializing CodeGenForCausalLM: ['transformer.h.0.attn.causal_mask', 'transformer.h.1.attn.causal_mask', 'transformer.h.10.attn.causal_mask', 'transformer.h.11.attn.causal_mask', 'transformer.h.12.attn.causal_mask', 'transformer.h.13.attn.causal_mask', 'transformer.h.14.attn.causal_mask', 'transformer.h.15.attn.causal_mask', 'transformer.h.16.attn.causal_mask', 'transformer.h.17.attn.causal_mask', 'transformer.h.18.attn.causal_mask', 'transformer.h.19.attn.causal_mask', 'transformer.h.2.attn.causal_mask', 'transformer.h.3.attn.causal_mask', 'transformer.h.4.attn.causal_mask', 'transformer.h.5.attn.causal_mask', 'transformer.h.6.attn.causal_mask', 'transformer.h.7.attn.causal_mask', 'transformer.h.8.attn.causal_mask', 'transformer.h.9.attn.causal_mask']
- This IS expected if you are initializing CodeGenForCausalLM from the checkpoint of a model trained on another task or with another architecture (e

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e7cb3c219274260b95.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
